# 💻 Unidad 2: Material Complementario - Práctica
## Módulo 01 - Visualizaciones Interactivas con Plotly
### Laboratorio - Universidad del Aconcagua

---

## 🎯 Objetivos

1. ✅ Crear 7 tipos de gráficos interactivos con Plotly
2. ✅ Personalizar layouts y temas
3. ✅ Implementar subplots y animaciones
4. ✅ Exportar y compartir visualizaciones

### 📋 Ejercicios

1. **Ejercicio 1**: Scatter plot con categorías
2. **Ejercicio 2**: Line chart temporal
3. **Ejercicio 3**: Bar chart comparativo
4. **Ejercicio 4**: Heatmap de correlación
5. **Ejercicio 5**: Subplots dashboard
6. **Ejercicio 6**: Animación temporal

---

## 🛠️ Setup

```python
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Cargar datos
ventas = pd.read_csv("/dbfs/FileStore/Laboratorio/Datasets/ventas.csv").head(1000)
print("✅ Datos cargados")
```

---

## 📊 Ejercicio 1: Scatter Plot Avanzado

```python
# Preparar datos
ventas_agg = ventas.groupby(['categoria', 'mes']).agg({
    'monto': 'sum',
    'cantidad': 'sum',
    'cliente_id': 'nunique'
}).reset_index()

# Crear scatter
fig = px.scatter(
    ventas_agg,
    x='cantidad',
    y='monto',
    color='categoria',
    size='cliente_id',
    hover_data=['mes'],
    title='Relación Cantidad-Monto por Categoría',
    labels={'cantidad': 'Cantidad Vendida', 'monto': 'Ingresos ($)'}
)

fig.update_layout(
    template='plotly_white',
    hovermode='closest'
)

fig.show()

print("✅ Scatter plot creado con interactividad")
```

---

## 📈 Ejercicio 2: Line Chart Temporal

```python
# Series temporales
ts = ventas.groupby('fecha')['monto'].sum().reset_index()
ts['fecha'] = pd.to_datetime(ts['fecha'])

fig = px.line(
    ts,
    x='fecha',
    y='monto',
    title='Evolución de Ventas Diarias'
)

# Personalizar
fig.update_traces(mode='lines+markers', line_color='#1f77b4')
fig.update_layout(
    xaxis_title='Fecha',
    yaxis_title='Ventas ($)',
    hovermode='x unified'
)

# Agregar línea de tendencia
fig.add_hline(
    y=ts['monto'].mean(),
    line_dash='dash',
    line_color='red',
    annotation_text='Promedio'
)

fig.show()

print("✅ Line chart con línea de referencia creado")
```

---

## 📊 Ejercicio 3: Bar Chart Comparativo

```python
# Comparación por categoría
cat_ventas = ventas.groupby(['categoria', 'mes'])['monto'].sum().reset_index()

fig = px.bar(
    cat_ventas,
    x='mes',
    y='monto',
    color='categoria',
    barmode='group',
    title='Ventas Mensuales por Categoría'
)

fig.update_layout(
    xaxis_title='Mes',
    yaxis_title='Ventas ($)',
    legend_title='Categoría',
    template='plotly_white'
)

fig.show()

print("✅ Bar chart agrupado creado")
```

---

## 🔥 Ejercicio 4: Heatmap Correlación

```python
# Calcular correlaciones
numeric_cols = ventas.select_dtypes(include=[np.number]).columns
corr_matrix = ventas[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    title='Matriz de Correlación'
)

fig.update_layout(
    width=800,
    height=800
)

fig.show()

print("✅ Heatmap de correlación creado")
```

---

## 📊 Ejercicio 5: Subplots Dashboard

```python
# Crear dashboard con 4 gráficos
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Ventas por Día', 'Top Productos', 
                    'Distribución Montos', 'KPI: Total'),
    specs=[[{'type': 'scatter'}, {'type': 'bar'}],
           [{'type': 'histogram'}, {'type': 'indicator'}]]
)

# 1. Line chart
fig.add_trace(
    go.Scatter(x=ts['fecha'], y=ts['monto'], mode='lines'),
    row=1, col=1
)

# 2. Bar chart
top_prod = ventas.groupby('producto')['monto'].sum().nlargest(5)
fig.add_trace(
    go.Bar(x=top_prod.index, y=top_prod.values),
    row=1, col=2
)

# 3. Histogram
fig.add_trace(
    go.Histogram(x=ventas['monto'], nbinsx=30),
    row=2, col=1
)

# 4. Indicator KPI
fig.add_trace(
    go.Indicator(
        mode='number+delta',
        value=ventas['monto'].sum(),
        delta={'reference': 100000, 'relative': True},
        title={'text': 'Total Ventas'}
    ),
    row=2, col=2
)

fig.update_layout(
    height=800,
    showlegend=False,
    title_text='Dashboard de Ventas'
)

fig.show()

print("✅ Dashboard con subplots creado")
```

---

## 🎥 Ejercicio 6: Animación Temporal

```python
# Preparar datos para animación
ventas['anio'] = pd.to_datetime(ventas['fecha']).dt.year
ventas['mes_num'] = pd.to_datetime(ventas['fecha']).dt.month

anim_data = ventas.groupby(['anio', 'mes_num', 'categoria']).agg({
    'monto': 'sum',
    'cantidad': 'sum'
}).reset_index()

fig = px.bar(
    anim_data,
    x='categoria',
    y='monto',
    color='categoria',
    animation_frame='mes_num',
    animation_group='categoria',
    range_y=[0, anim_data['monto'].max() * 1.1],
    title='Evolución Mensual de Ventas por Categoría'
)

fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 500

fig.show()

print("✅ Animación temporal creada")
print("💡 Usa los controles play/pause para ver la evolución")
```

---

## 💾 Export y Compartir

```python
# Guardar visualizaciones
fig.write_html('/dbfs/FileStore/Laboratorio/reportes/dashboard_ventas.html')
print("✅ Dashboard guardado como HTML")

# También como imagen (requiere kaleido)
try:
    fig.write_image('/dbfs/FileStore/Laboratorio/reportes/dashboard.png', width=1200, height=800)
    print("✅ Imagen PNG exportada")
except:
    print("⚠️ Instala kaleido para exportar imágenes: %pip install kaleido")
```

---

## ✅ Resumen

### 💡 Aprendizajes

**Visualizaciones creadas:**
* ✅ Scatter plot con categorías y tamaños
* ✅ Line chart con líneas de referencia
* ✅ Bar chart agrupado y apilado
* ✅ Heatmap de correlación
* ✅ Dashboard con subplots
* ✅ Animación temporal interactiva

**Características interactivas:**
* ✅ Zoom, pan, hover tooltips
* ✅ Selección de leyenda
* ✅ Controles de animación
* ✅ Export HTML/PNG

---

### 🚀 Próximos Pasos

* Continúa con **Módulo 02: Mapas y Visualización Geoespacial**
* Integra visualizaciones en dashboards del Módulo 04
* Aplica en tus TPs

---

**Universidad del Aconcagua - Mendoza 🇦🇷**